# Лабораторная работа 03. Сводный анализ данных в Pandas

**Курс:** «Анализ данных и искусственный интеллект», 1 курс бакалавриата.

**Задача аналитика:** изучить предложения автомобилей на вторичном рынке и подготовить основу для оценки цены.

После работы вы сможете рассчитывать описательные статистики, сравнивать группы, строить сводные таблицы и простые графики средствами Pandas.

**План:** структура данных → статистики → `groupby` → `pivot_table` и доли → простая оценка цены → `Pandas.plot`.
Расширенная визуализация и бизнес-представление результатов вынесены в лабораторную 04.

Основные разделы рассчитаны на одну практику; проверка оценки цены на новых данных — дополнительное задание.


Разведочный анализ данных (EDA) помогает понять набор данных до построения модели: какие объекты представлены, какие значения типичны и какие записи требуют проверки. В этой работе основной инструмент — таблица, а простые графики помогают проверить выводы.

Одна строка описывает автомобиль в наборе предложений. `Price` — указанная цена в **евро**, `Year` — **год выпуска**, `Distance` — пробег в километрах. В данных нет даты продажи, факта сделки или времени нахождения объявления на сайте. Поэтому число записей характеризует представленность в выборке, а не спрос; указанная цена не обязательно равна цене сделки.


## Что должен выяснить аналитик

1. Какие марки и сегменты хорошо представлены в выборке?
2. Какая цена типична для каждого сегмента и насколько велик разброс?
3. Где данных мало и выводы требуют осторожности?
4. Может ли средняя или медианная цена похожих автомобилей служить простой оценкой?

Редкое значение не обязательно является ошибкой. Запись сначала проверяют по нескольким признакам, затем обосновывают решение о её обработке.


## 1. Загрузка данных


In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Положите CSV рядом с ноутбуком. Для примера из репозитория — резервный адрес.
DATA_FILE = 'cars_processed.csv'
data_path = Path(DATA_FILE)
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(
        'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/'
        'refs/heads/main/Workshops/cars_processed.csv'
    )

# Совместимость с результатом предыдущей практики.
df = df.rename(columns={'Price(euro)': 'Price', 'Engine_capacity(cm3)': 'Engine'})
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed:')])

if 'Transmission' not in df:
    df['Transmission'] = df['AutoTransmission'].map({True: 'Automatic', False: 'Manual'})


num_columns = ['Year', 'Distance', 'Engine', 'Price', 'km_year']
cat_columns = ['Make', 'Model', 'Style', 'Fuel_type', 'Transmission']
df = df[num_columns + cat_columns]
df.head()


,Year,Distance,Engine,Price,km_year,Make,Model,Style,Fuel_type,Transmission
0,2011,195000.0,1800.0,7750.0,17727.272727,Toyota,Prius,Hatchback,Hybrid,Automatic
1,2014,135000.0,1500.0,8550.0,16875.000000,Renault,Grand Scenic,Universal,Diesel,Manual
2,2012,110000.0,1500.0,6550.0,11000.000000,Renault,Laguna,Universal,Diesel,Manual
3,2006,200000.0,1600.0,4100.0,12500.000000,Opel,Astra,Universal,Metan/Propan,Manual
4,2000,300000.0,2200.0,3490.0,13636.363636,Mercedes,Vito,Microvan,Diesel,Manual


### Входной файл


Используйте `cars_processed.csv` из лабораторной 02 или свой файл с теми же колонками. `cars_cleaned.csv` — более поздний результат очистки; для воспроизведения примеров он не нужен. Исходный CSV не перезаписывается.


## 2. Первичная проверка

Проверим размер таблицы, типы колонок, пропуски и точные дубликаты. До удаления дубликатов запишем их число. Одинаковые описания разных объявлений могут совпадать; здесь удаление точных повторов используется как учебное допущение.


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32067 entries, 0 to 32066
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Year          32067 non-null  int64  
 1   Distance      32067 non-null  float64
 2   Engine        32067 non-null  float64
 3   Price         32067 non-null  float64
 4   km_year       32067 non-null  float64
 5   Make          32067 non-null  object 
 6   Model         32067 non-null  object 
 7   Style         32067 non-null  object 
 8   Fuel_type     32067 non-null  object 
 9   Transmission  32067 non-null  object 
dtypes: float64(4), int64(1), object(5)
memory usage: 2.4+ MB


In [16]:
df.shape

(32067, 10)

2. Вспомним про то, как проверить данные на количество пропусков и дубликатов

In [17]:
df.isna().sum().sum() # Пропуски

np.int64(0)

In [18]:
df.duplicated().sum()

np.int64(1)

In [21]:
df = df.drop_duplicates().copy()

## 3. Сводный анализ признаков


Во многих случаях аналитики должны предоставить агрегированные или сгруппированные данные. Эта информация помогает понять общие характеристики в определенных разрезах данных. Часто этого достаточно, чтобы достичь бизнес-цели или принимать решения.

Также сводный анализ позволяет достаточно просто нати наиболее редкие записи данных или наиболее редкие комбинации данные

### Описательные статистики

Попробуем проанализировать данные 

1. посмотрим на то, какие у нас есть категориальные и численные столбцы.
   
[Напомним, что категориальные переменные](https://education.yandex.ru/handbook/data-analysis/article/rabota-s-kategorialnymi-dannymi-v-python) — это те переменные, которые выражены ограниченным диапазоном значений. Например, группа крови или семейное положение.



In [22]:
display(pd.DataFrame({'Численные': pd.Series(num_columns),
                      'Категориальные': pd.Series(cat_columns)}))

,Численные,Категориальные
0,Year,Make
1,Distance,Model
2,Engine,Style
3,Price,Fuel_type
4,km_year,Transmission


Статистика по числовым данным

In [30]:
# pd.set_option('display.Year', 0)
df.describe(percentiles = [0.01, 0.05,0.25,0.50,0.75,0.95, 0.99]).round(2)


,Year,Distance,Engine,Price,km_year
count,32066.00,32066.00,32066.00,32066.00,32066.00
mean,2008.23,181089.63,1867.13,9561.38,14894.21
std,7.39,99426.96,610.51,10615.28,8019.99
min,1953.00,0.00,200.00,200.00,0.00
1%,1987.00,1800.00,200.00,650.00,94.86
5%,1994.00,20449.75,1200.00,1300.00,1848.86
25%,2004.00,121000.00,1500.00,3500.00,10013.75
50%,2009.00,180000.00,1800.00,6700.00,14400.00
75%,2014.00,236000.00,2000.00,12000.00,18846.15
95%,2018.00,340998.00,3000.00,26899.75,28230.56


Прочитайте `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`.
`50%` — медиана; между `25%` и `75%` лежит центральная половина наблюдений.
Среднее чувствительно к крайним значениям. Максимум сам по себе не доказывает ошибку.

**Вопросы:** насколько различаются средняя и медианная цена? Какие максимальные значения нужно проверить? Почему квартиль года выпуска не является основанием удалять более старые автомобили?


Аналогично примеру выше посчитаем статистик по категориальным признакам. Это будут признаки с указанием типов данных `object` и `bool`

In [33]:
df[cat_columns].astype('object').describe()


,Make,Model,Style,Fuel_type,Transmission
count,32066,32066,32066,32066,32066
unique,22,789,12,6,2
top,Volkswagen,E Class,Sedan,Diesel,Manual
freq,3403,1094,9737,15141,17474


In [35]:
df[list(cat_columns) + ['Year']].astype('object').describe()

,Make,Model,Style,Fuel_type,Transmission,Year
count,32066,32066,32066,32066,32066,32066
unique,22,789,12,6,2,65
top,Volkswagen,E Class,Sedan,Diesel,Manual,2011
freq,3403,1094,9737,15141,17474,2060


Другим важным инструментом анализа является группировка данных. 

В общем случае группировка данных в Pandas выглядит следующим образом:
```python
df.groupby(by=grouping_columns)[columns_to_show].function()
```

1. К датафрейму применяется метод `groupby`, который групперует данные по `grouping_columns` – признаку или набору признаков.
2. Выбираем нужные нам столбцы (`columns_to_show`).
3. К полученным группам применяется функция или несколько функций.

Попробуем сгруппировать данные по производителю ('Make'). Посмотрим сколько автомобилей кажого производителя имеется по годам. В качестве функции группировки будем использовать подсчет значений в группе (`count`), минимальный и максимальный год автомобиля в группе `np.min, np.max`. Для того, чтобы использовать сразу 3 функции вместе воспользуемся методом `agg`

In [34]:
df.groupby(['Make','Year'])[['Price']].mean()

Price
Make  Year              
Audi  1980   1500.000000
      1982    650.000000
      1983    970.000000
      1984   1020.000000
      1985    950.000000
...                  ...
Volvo 2017  27917.027778
      2018  30726.888889
      2019  42709.666667
      2020  49216.166667
      2021  58600.000000

[797 rows x 1 columns]

In [36]:
df.groupby(['Make','Year'])['Price'].agg(['mean', 'count'])

mean  count
Make  Year                     
Audi  1980   1500.000000      1
      1982    650.000000      1
      1983    970.000000      5
      1984   1020.000000      5
      1985    950.000000      1
...                  ...    ...
Volvo 2017  27917.027778     36
      2018  30726.888889     18
      2019  42709.666667      9
      2020  49216.166667      6
      2021  58600.000000      2

[797 rows x 2 columns]

### Упражнение

Выведите 10 марок с самой высокой средней ценой. Затем проверьте, сколько автомобилей приходится на каждую марку: рейтинг по одной дорогой записи может быть нестабильным.


In [37]:
df.groupby('Make')['Price'].mean().sort_values(ascending=False).head(10)

Make
Lexus       18517.707692
Volvo       15549.217770
BMW         14005.245998
Mercedes    13491.040973
Audi        12791.282552
KIA         12338.304615
Hyundai     11809.161236
Rare        10828.207787
Toyota       9696.309193
Honda        9285.007839
Name: Price, dtype: float64

### Усиление 1. Среднее, медиана и размер группы

`agg` рассчитывает несколько показателей за один вызов. `size` считает строки, `count` — известные значения выбранной колонки. Именованная агрегация позволяет дать результатам понятные названия.



In [40]:
make_summary = df.groupby('Make').agg(
    n=('Price', 'size'), known_price=('Price', 'count'),
    mean_price=('Price', 'mean'), median_price=('Price', 'median'),
    q1=('Price', lambda s: s.quantile(0.25)),
    q3=('Price', lambda s: s.quantile(0.75))
)

display(make_summary.sort_values('n', ascending=False).head(10).round(1))


,n,known_price,mean_price,median_price,q1,q3
Make,,,,,,
Volkswagen,3403,3403,6923.4,5900.0,3200.0,8900.0
Toyota,3024,3024,9696.3,7754.0,5500.0,11500.0
Mercedes,3002,3002,13491.0,8900.0,4299.0,17500.0
Rare,2594,2594,10828.2,5100.0,2600.0,12350.0
BMW,2561,2561,14005.2,9400.0,4500.0,18999.0
Renault,2334,2334,6649.2,6199.0,3500.0,8499.0
Opel,1837,1837,4175.7,3650.0,1900.0,5600.0
Skoda,1702,1702,8920.9,6999.0,4950.0,10786.8
Audi,1536,1536,12791.3,9299.5,3500.0,17500.0


**Задание:** сравните рейтинг по средней цене и по медиане. Найдите марку, для которой вывод изменился. Укажите размер группы и объясните расхождение. Ни один рейтинг не доказывает, что марка выгоднее для перепродажи: затрат и цен сделок в данных нет.


## 4. Построение сводных таблиц


Анализ при помощи сводных таблиц называется __pivot_table__

В качестве примера анализ __PIVOT__ рассмотрим задачу оценки популярность типов кузова по маркам автомобилей.

* `data=df` - Исходные данные
* `index='Make'` - Группировка по марке — будет в строках
* `columns='Style'` - Группировка по типу кузова — будет в столбцах
* `aggfunc='size'` - Функция агрегации: считает количество строк в каждой группе
* `fill_value=0` - Заменяет отсутствующие значения (`NaN`) на `0`

In [52]:
pivot_style = pd.pivot_table(df, index=['Make','Year'], columns=['Style'] , values = 'Price',  aggfunc='size', fill_value=0)
# pivot_style = pd.pivot_table(df, index=['Make'], columns=['Style'] ,  aggfunc='size', fill_value=0)
pivot_style

Style       Cabriolet  Combi  Coupe  Crossover  Hatchback  Microvan  Minivan  \
Make  Year                                                                     
Audi  1980          0      0      0          0          1         0        0   
      1982          0      0      0          0          0         0        0   
      1983          0      0      0          0          0         0        0   
      1984          0      0      0          0          0         0        0   
      1985          0      0      0          0          0         0        0   
...               ...    ...    ...        ...        ...       ...      ...   
Volvo 2017          0      0      0         11          1         0        0   
      2018          0      0      0          8          0         0        0   
      2019          0      0      0          3          0         0        0   
      2020          0      0      0          3          0         0        0   
      2021          0      0      0          1          0         0        0   

Style       Pickup  Roadster  SUV  Sedan  Universal  
Make  Year                                           
Audi  1980       0         0    0      0          0  
      1982       0         0    0      1          0  
      1983       0         0    0      4          1  
      1984       0         0    0      4          1  
      1985       0         0    0      1          0  
...            ...       ...  ...    ...        ...  
Volvo 2017       0         0   11      9          4  
      2018       0         0    4      5          1  
      2019       0         0    2      4          0  
      2020       0         0    2      1          0  
      2021       0         0    1      0          0  

[797 rows x 12 columns]

In [54]:
pd.pivot_table(df, index=['Make','Year'], columns=['Style'] , values = 'Price',  aggfunc='mean', fill_value=0).round(1)

Style       Cabriolet  Combi  Coupe  Crossover  Hatchback  Microvan  Minivan  \
Make  Year                                                                     
Audi  1980        0.0    0.0    0.0        0.0     1500.0       0.0      0.0   
      1982        0.0    0.0    0.0        0.0        0.0       0.0      0.0   
      1983        0.0    0.0    0.0        0.0        0.0       0.0      0.0   
      1984        0.0    0.0    0.0        0.0        0.0       0.0      0.0   
      1985        0.0    0.0    0.0        0.0        0.0       0.0      0.0   
...               ...    ...    ...        ...        ...       ...      ...   
Volvo 2017        0.0    0.0    0.0    26022.6    18800.0       0.0      0.0   
      2018        0.0    0.0    0.0    28335.9        0.0       0.0      0.0   
      2019        0.0    0.0    0.0    52299.3        0.0       0.0      0.0   
      2020        0.0    0.0    0.0    52166.0        0.0       0.0      0.0   
      2021        0.0    0.0    0.0    47700.0        0.0       0.0      0.0   

Style       Pickup  Roadster      SUV    Sedan  Universal  
Make  Year                                                 
Audi  1980     0.0       0.0      0.0      0.0        0.0  
      1982     0.0       0.0      0.0    650.0        0.0  
      1983     0.0       0.0      0.0    875.0     1350.0  
      1984     0.0       0.0      0.0   1050.0      900.0  
      1985     0.0       0.0      0.0    950.0        0.0  
...            ...       ...      ...      ...        ...  
Volvo 2017     0.0       0.0  33020.5  26509.8    24537.5  
      2018     0.0       0.0  34000.0  31479.4    33000.0  
      2019     0.0       0.0  39945.0  36899.8        0.0  
      2020     0.0       0.0  59449.5  19900.0        0.0  
      2021     0.0       0.0  69500.0      0.0        0.0  

[797 rows x 12 columns]

In [42]:
pivot_style = pd.pivot_table(
    df, index=['Make','Year'], columns=['Style'],
    aggfunc='size', fill_value=0
)

# Сумма по каждому кузову
style_totals = pivot_style.sum(axis=0).sort_values(ascending=False)
print(style_totals)

# Самый популярный
top_style = style_totals.idxmax()
print("Самый популярный кузов:", top_style)

Style
Sedan        9737
Universal    5805
Hatchback    5195
Crossover    4042
SUV          3046
Minivan      2631
Coupe         574
Microvan      417
Combi         341
Cabriolet     131
Pickup        124
Roadster       23
dtype: int64
Самый популярный кузов: Sedan


К результатам pivot можно обращаться через методы `loc` и `iloc`

Попробуем теперь посмотреть сводку по конкретным маркам 

In [44]:
pivot_style.loc[['Toyota']]

Style        Cabriolet  Combi  Coupe  Crossover  Hatchback  Microvan  Minivan  \
Make   Year                                                                     
Toyota 1986          0      0      0          0          0         0        0   
       1988          0      0      0          0          0         0        0   
       1989          0      0      0          0          1         0        0   
       1990          0      0      0          0          0         0        0   
       1991          0      0      0          0          1         0        1   
       1992          0      0      0          0          1         0        0   
       1993          0      0      1          0          4         0        0   
       1994          0      0      1          0          2         0        1   
       1995          0      0      0          0          0         0        0   
       1996          0      0      0          2          3         0        0   
       1997          0      0      0          1          2         0        1   
       1998          0      0      0          0          2         0        2   
       1999          0      0      0          2          3         0        1   
       2000          0      0      0          1          2         0        4   
       2001          0      0      1          3          2         0        7   
       2002          0      0      0          3          7         0       11   
       2003          0      0      0          7         17         4        9   
       2004          0      0      4          3         19         1       23   
       2005          0      1      4          7         71         4       25   
       2006          0      3      1         21         68         3       12   
       2007          0      2      2         16         97         0       10   
       2008          0      0      0         14         77         1        4   
       2009          0      0      1         21         69         1       12   
       2010          0      1      1         17         95         1        9   
       2011          0      1      0          9        116         2        7   
       2012          0      0      2         10         77         1        3   
       2013          0      1      0         21         74         3       11   
       2014          0      0      0         14         42         3        5   
       2015          0      2      0         14         42         1        9   
       2016          0      0      0         16         38         0        2   
       2017          0      0      0         43         26         0        0   
       2018          0      0      0         25         14         0        2   
       2019          0      0      0         11          6         0        0   
       2020          0      0      0          7          0         0        0   
       2021          0      0      0          6          0         0        2   

Style        Pickup  Roadster  SUV  Sedan  Universal  
Make   Year                                           
Toyota 1986       0         0    0      1          0  
       1988       0         0    0      2          0  
       1989       0         0    0      0          0  
       1990       0         0    0      1          0  
       1991       0         0    1      2          0  
       1992       0         0    0      1          0  
       1993       0         0    3      3          1  
       1994       0         0    0      4          1  
       1995       0         0    0      3          2  
       1996       0         0    2      4          1  
       1997       0         0    1      1          2  
       1998       0         0    1      5          4  
       1999       0         0    1      2          1  
       2000       0         0    1     11          1  
       2001       0         0    3      7          6  
       2002       0         0    6     10          6  
       2003 

Определим сколько у 'Toyota' записей 'Sedan'

In [45]:
pivot_style.loc['Toyota', 'Sedan']

Year
1986     1
1988     2
1989     0
1990     1
1991     2
1992     1
1993     3
1994     4
1995     3
1996     4
1997     1
1998     5
1999     2
2000    11
2001     7
2002    10
2003    18
2004    22
2005    34
2006    27
2007    64
2008    62
2009    43
2010    34
2011    19
2012    43
2013    43
2014    42
2015    44
2016    48
2017    26
2018    13
2019    11
2020     5
2021     3
Name: Sedan, dtype: int64

Посчитаем общие количество кузовов кажого типа

In [46]:
pivot_style.sum(axis=0)

Style
Cabriolet     131
Combi         341
Coupe         574
Crossover    4042
Hatchback    5195
Microvan      417
Minivan      2631
Pickup        124
Roadster       23
SUV          3046
Sedan        9737
Universal    5805
dtype: int64

Или отсортеруем сводку по тому у какой марке больше всего седанов

In [47]:
pivot_style.sort_values(by='Sedan',ascending = False)

Style            Cabriolet  Combi  Coupe  Crossover  Hatchback  Microvan  \
Make       Year                                                            
Mercedes   2014          1      0      5         31          9         4   
Volkswagen 2013          0      4      0         15         23         4   
BMW        2006          3      1      5         19          3         0   
           2004          3      0      5         16          2         0   
           2005          0      0      2         13          3         0   
...                    ...    ...    ...        ...        ...       ...   
Chevrolet  1998          0      0      1          0          0         1   
Dacia      1999          0      1      0          0          0         0   
Volvo      1992          0      0      0          0          0         0   
Audi       1980          0      0      0          0          1         0   
Volvo      2021          0      0      0          1          0         0   

Style            Minivan  Pickup  Roadster  SUV  Sedan  Universal  
Make       Year                                                    
Mercedes   2014       17       0         0   13     92         13  
Volkswagen 2013       26       4         0    9     90         33  
BMW        2006        0       0         0   17     88          5  
           2004        0       0         0   17     87         11  
           2005        1       0         0   30     84         10  
...                  ...     ...       ...  ...    ...        ...  
Chevrolet  1998        0       1         0    0      0          0  
Dacia      1999        0       0         0    0      0          0  
Volvo      1992        0       0         0    0      0          1  
Audi       1980        0       0         0    0      0          0  
Volvo      2021        0       0         0    1      0          0  

[797 rows x 12 columns]

Сводка может производится по нескольким параметрам. Например с целью определить аномалии проверим в каких марках и по каким годам распределены кузова

In [48]:
pivot_style = pd.pivot_table(df, index=['Make',], columns='Style', aggfunc='size', fill_value=0)
pivot_style

Style,Cabriolet,Combi,Coupe,Crossover,Hatchback,Microvan,Minivan,Pickup,Roadster,SUV,Sedan,Universal
Make,,,,,,,,,,,,
Audi,10,8,54,185,63,1,7,0,1,100,796,311
BMW,13,4,84,354,79,5,17,1,0,305,1520,179
Chevrolet,2,0,7,42,65,2,12,3,0,27,142,20
Dacia,0,21,1,175,142,10,144,6,1,105,343,401
Ford,4,23,27,100,276,19,134,37,2,49,362,298
Honda,0,0,18,237,153,3,16,0,0,134,298,34
Hyundai,0,3,15,372,189,7,53,1,1,272,226,58
KIA,0,2,5,151,136,7,52,0,1,129,111,56
Lexus,3,0,5,141,109,1,0,0,1,113,203,9


In [49]:
pivot_style = pd.pivot_table(df, index=['Make',], columns='Year', aggfunc='size', fill_value=0)
pivot_style

Year,1953,1954,1955,1958,1959,1960,1961,1962,1963,1964,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
Make,,,,,,,,,,,,,,,,,,,,,
Audi,0,0,0,0,0,0,0,0,0,0,...,113,100,67,74,88,60,48,15,10,6
BMW,0,0,0,0,0,0,0,0,0,0,...,123,106,134,164,133,122,67,32,24,9
Chevrolet,0,0,0,0,0,0,0,0,0,0,...,37,19,26,11,20,18,25,5,2,0
Dacia,0,0,0,0,0,0,0,0,0,0,...,159,128,127,88,61,57,44,23,9,18
Ford,0,0,0,0,0,0,0,0,0,0,...,62,109,79,127,152,126,72,25,6,0
Honda,0,0,0,0,0,0,0,0,0,0,...,26,48,46,40,33,27,52,20,13,5
Hyundai,0,0,0,0,0,0,0,0,0,0,...,60,86,81,90,122,98,62,46,29,10
KIA,0,0,0,0,0,0,0,0,0,0,...,45,35,40,40,43,76,36,21,33,16
Lexus,0,0,0,0,0,0,0,0,0,0,...,40,66,47,69,64,26,23,11,2,3


Отметим, что аналогичных результатов `PIVOT` можно получить несколькими способами:
1. `groupby()` - более гибкий, позволяет добавлять дополнительные операции или делать более детальную группировку, но это менее читаемый способ. 
```python
groupby_style = df.groupby(['Make', 'Year', 'Style']).size().unstack(fill_value=0) #groupby c использованием unstack
```

2. `crosstab()` - [простой, но не гибкий сопособ подсчета значений](https://sky.pro/wiki/analytics/analiz-dannyh-s-pdcrosstab-v-python-sozdaem-svodnye-tablicy)
```python
crosstab_style = pd.crosstab(index=[df['Make'], df['Year']], columns=df['Style'])
```

3. `pivot_table()` - способ агрегации по значениям, а не просто подсчет значений
```python
pivot_style = pd.pivot_table(df, index=['Make','Year'], columns='Style', aggfunc='size', fill_value=0)
```
   

### Усиление 2. Число записей и доля — разные вопросы

`crosstab` считает сочетания категорий. `normalize='index'` делит каждую строку на её сумму: получаем структуру кузовов **внутри марки**.

В таблице цен отсутствие наблюдений оставляем как `NaN`. Ноль означал бы известную нулевую цену.


In [ ]:
style_count = pd.crosstab(df['Make'], df['Style'])
style_share = 100 * pd.crosstab(df['Make'], df['Style'], normalize='index')
top_makes = df['Make'].value_counts().head(10).index
display(style_count.loc[top_makes])
display(style_share.loc[top_makes].round(1))
price_pivot = df.pivot_table(index='Make', columns='Transmission',
                             values='Price', aggfunc='median')
display(price_pivot.loc[top_makes].round(0))


**Задание:** найдите марку с наибольшим числом SUV и марку с наибольшей долей SUV среди марок с хотя бы 30 записями. Совпадают ли ответы? Укажите знаменатель каждой доли.


### Упражнения: редкие комбинации

1. Найдите редкие сочетания марки и кузова. Для трёх примеров изучите исходные записи и предложите решение: оставить, уточнить у источника или исправить подтверждённую ошибку.
2. Обсудите объединение `Roadster` и `Cabriolet` в аналитическую группу. Это изменение детализации отчёта, а не исправление ошибки. Сохраните исходный `Style`.

Чистота данных помогает анализу, но равное число объектов в группах не гарантирует лучшую модель. Объединение категорий требует предметного обоснования.


In [ ]:
df['Style_group'] = df['Style'].replace(
    {'Roadster': 'Cabriolet/Roadster', 'Cabriolet': 'Cabriolet/Roadster'}
)


In [ ]:
pd.crosstab(df['Make'], df['Style_group'])


3. Предложите заполнение пропусков в численной колонке медианой похожих автомобилей. Выберите признаки группировки и правило для группы, где нет известных значений. В упражнении скрывайте только известные значения входного признака в копии таблицы, чтобы оценить ошибку заполнения.

Не заменяйте дорогой автомобиль средней ценой только из-за редкости. `Price` — целевая переменная: её искусственное заполнение создаёт выдуманные ответы для будущей модели.


4. Для каждой марки определите самый частый кузов. Обсудите заполнение **пропущенного**, а не просто редкого кузова модой группы. Как поступить при равных частотах и неизвестной марке?


5. Постройте таблицу числа автомобилей по маркам и десятилетиям **выпуска**. Используйте `(Year // 10) * 10`. Это распределение по возрастным когортам; для `resample` нужен временной индекс, которого здесь нет.


In [ ]:
df_numeric_and_make = df[num_columns + ['Make']].copy()
df_numeric_and_make['Decade'] = (df_numeric_and_make['Year'] // 10) * 10
pivot_make = pd.pivot_table(df_numeric_and_make, index='Make',
                            columns='Decade', aggfunc='size', fill_value=0)
pivot_make.sum()


## 5. Простая оценка цены по группе

Средняя цена автомобилей той же марки и года выпуска — простой ориентир. Он не учитывает модель, комплектацию и состояние. Построим такую таблицу и измерим разброс цен вокруг группового среднего.

$$\mathrm{MAE}=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

Здесь $y_i$ — указанная цена автомобиля, $\hat y_i$ — оценка. MAE выражается в евро.
На тех же данных, из которых рассчитаны средние, MAE описывает внутригрупповое отклонение; качество прогноза новых автомобилей требует отдельной проверки.


In [ ]:
df_numeric_and_make = df[list(num_columns) + ['Make']]
pivot_make = pd.pivot_table(df_numeric_and_make[['Make','Year','Price']], 
                           index=['Make'], 
                           columns='Year', 
                           values='Price',  # явно указываем колонку для агрегации
                           aggfunc='mean')  # используем строку вместо np.mean для наглядности

проверим как это работает для одного производителя в заданный год. Года будем выбирать в диапазоне от 2010 до 2020, из анализа выше видно, что это наиболее распростроненный период.

In [ ]:
make = 'Audi'
years = range(2010, 2020)
year = 2015
year


In [ ]:
price_mean = pivot_make.loc[make, year]
print(f"Средняя цена {make} в {year} году: {float(price_mean):.2f}")

In [ ]:
price_mean

In [ ]:
price_by_year = df[(df['Make'] == make) & (df['Year']==year)]['Price']
mae = (price_by_year - price_mean).abs().mean()
print(f"Cреднее абсолютное отклонение (MAE) для оценки по pivot автомобиля {make} в {year} году: {mae:.2f}")

Подсчитаем значение ошибки MAE за заданный период для заданного производителя. Для этого зададим фильтр данных и создадим в нем новый столбец `mean_by_year`

In [ ]:
# выбираем часть набора данных по производитель - диапазон year
filtered_df = df[(df['Make'] == make) & (df['Year'].isin(years))].copy()
#заполняем столбец в формате год - средняя цена
filtered_df['mean_by_year'] = filtered_df['Year'].map(pivot_make.loc[make]) 


mae = (filtered_df['Price']-filtered_df['mean_by_year']).abs().mean()

print(f'Cреднее абсолютное отклонение (MAE) оценки по pivot для автомобилей {make} {years[0]} - {years[-1]} года = {mae:.1f}  ')

Также на самом деле можно бы было подсчитать среднее MAE для всего набора данных

воспользуемся неочевидным вариантом функция `transform` применяет функцию к серии и возвращает массив исходной размерности. 

In [ ]:
mean_by_group = df.groupby(['Make', 'Year'])['Price'].transform('mean')
mae = (mean_by_group - df['Price']).abs().mean()
print(f'Внутригрупповое среднее абсолютное отклонение: {mae:.1f} евро')


Среднее группы содержит цену самого оцениваемого автомобиля. У группы из одной записи отклонение равно нулю — это не свидетельство точного прогноза.

**Упражнения:** сравните среднее и медиану группы; добавьте `Model`; объясните, почему дробление групп снижает отклонение на исходных данных, но может ухудшить оценку новых автомобилей.

Дополнительно рассчитайте MAPE для положительных цен. Низкие цены имеют большой вес в процентной ошибке, поэтому приводите MAPE вместе с MAE.


## 6. Простые графики Pandas


### От сводной таблицы к графику


`Series.plot` и `DataFrame.plot` позволяют быстро показать результат группировки. Отдельно изучать API Matplotlib в этой работе не требуется, но библиотека Matplotlib должна быть установлена.

Покажем среднюю цену Mercedes по **годам выпуска**. Линия связывает разные группы автомобилей в одной выборке; она не показывает изменение рыночной цены во времени.


In [ ]:
make = 'Mercedes'
filter_df = df[df['Make'] ==make]
filter_df_by_year = filter_df.groupby('Year')['Price'].mean()

In [ ]:
filter_df_by_year.plot();


Обратите внимание, что результат groupby имеет индексты - года!

In [ ]:
filter_df_by_year.index

Сделаем более приятный для чтения график

In [ ]:
filter_df_by_year.plot(
    marker='o', figsize=(12, 3),
    title=f'Средняя цена {make} по годам выпуска',
    xlabel='Год выпуска', ylabel='Средняя цена, евро',
    color='tab:blue', grid=True
);


Тот же результат можно представить столбчатой диаграммой. Сравните читаемость при большом числе годов. Выберите вариант, который лучше отвечает на ваш вопрос.


In [ ]:
filter_df_by_year.plot(
    kind='bar', figsize=(12, 3),
    title=f'Средняя цена {make} по годам выпуска',
    xlabel='Год выпуска', ylabel='Средняя цена, евро', color='tab:blue'
);


## 7. Самостоятельная работа и результат

Подготовьте небольшой табличный отчёт:

1. Пять показателей набора: число записей, медиана цены, квартильный диапазон цены, медиана пробега, число марок.
2. Сравнение не менее трёх сегментов с ценой, разбросом и числом записей.
3. Две сводные таблицы: абсолютные количества и доли. Объясните различие выводов.
4. Два графика `Pandas.plot` с единицами измерения и подписью, объясняющей результат.
5. Три вывода: наблюдение в числах → возможное объяснение → ограничение вывода.

**Что сдавать:** выполненный ноутбук. Не включайте среднюю цену группы, рассчитанную по всей таблице, во входные признаки будущей модели: она использует целевую переменную.

Сводные таблицы показали состав выборки и типичные цены сегментов. Следующий шаг — визуально проверить распределения, сравнить сегменты и представить результаты для заказчика в лабораторной 04.


### Передача данных в лабораторную 04

Сохраняем исходные признаки и учебные категории без оценок, рассчитанных из цены. Новый файл не заменяет CSV из лабораторной 02.


In [ ]:
export_columns = [c for c in df.columns if c != 'mean_make_price_by_year']
df[export_columns].to_csv('cars_summary_input.csv', index=False)
make_summary.to_csv('cars_make_summary.csv', index=True)


## Дополнительно. Проверка простой оценки на отложенных данных

Это задание можно выполнить после основного отчёта. Разделите строки случайно в отношении 80/20 с фиксированным seed. Таблицу средних стройте только на первой части. Для неизвестной группы используйте среднюю цену первой части и отдельно сообщайте долю таких случаев. Сравните MAE с общей средней ценой.

Так проверяется перенос на другие строки той же выборки. Для проверки будущего рынка потребовались бы даты объявлений; повторные объявления одного автомобиля также могут завысить результат случайного разделения.


In [ ]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index).copy()
group_mean = train.groupby(['Make', 'Year'])['Price'].mean().rename('estimate')
test = test.join(group_mean, on=['Make', 'Year'])
unknown = test['estimate'].isna()
fallback = train['Price'].mean()
test['estimate'] = test['estimate'].fillna(fallback)
print(f'Новых групп: {unknown.mean():.1%}')
print(f'MAE общей средней: {(test["Price"] - fallback).abs().mean():.1f} евро')
print(f'MAE групповой оценки: {(test["Price"] - test["estimate"]).abs().mean():.1f} евро')


Документация: [группировка и агрегация Pandas](https://pandas.pydata.org/docs/user_guide/groupby.html).
